# Wikipedia RAG Indexing Pipeline (FAISS)

Creates a FAISS index for RAG evaluation with popularity metadata.

## Advantages over Elasticsearch version:
- No external service required — pure in-process
- Simple persistence via `save_local()` / `load_local()`
- Supports exact (IndexFlat) and approximate (HNSW) vector search
- BM25 via `rank_bm25` for lexical retrieval
- Hybrid search via Reciprocal Rank Fusion (RRF)

## Prerequisites:
```bash
pip install -qU langchain-community faiss-cpu rank-bm25
```

## Steps
1. Load QA datasets from HuggingFace
2. Load Wikipedia corpus
3. Add popularity metadata
4. Create FAISS index (supports vector, approximation, bm25, and hybrid)

In [1]:
from pathlib import Path
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
from rag.faiss_rag_service import FaissRagService
from datasets import load_dataset, concatenate_datasets
from rag.utils import IndexingConfig
from config import DATA_DIR, CACHE_DIR
import numpy as np
from tqdm import tqdm
import logging
import dotenv
import os
import gc

dotenv.load_dotenv()

# Suppress noisy HTTP logs from libraries
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)

# ============================================================================

# ── FAISS ────────────────────────────────────────────────────────────────────

STRATEGY = "vector"          # "vector", "hnsw", "ivfpq"

# ── Datasets ─────────────────────────────────────────────────────────────────
QA_DATASETS = []
WIKIPEDIA_DATASET = "facebook/kilt_wikipedia"
WIKIPEDIA_VERSION = "2019-08-01"
POPULARITY_DATASET = "Cyro1/enwiki_pageviews_m"

# ── Index naming & paths ─────────────────────────────────────────────────────
NAME = "wiki_full_f"
COLLECTION_NAME = NAME
N_RANDOM_SAMPLES = None
COLLECTION_ROOT = Path(DATA_DIR) / NAME
QUESTIONS_PATH = COLLECTION_ROOT / "train_questions.parquet"
WIKI_PARQUET_PATH = COLLECTION_ROOT / "wiki_corpus.parquet"
FAISS_INDEX_DIR = COLLECTION_ROOT / "faiss_index"   # persistence dir

# ── Embedding (runtime — no Modal redeploy needed) ──────────────────────────
EMBEDDING_PROVIDER = "modal"       # "modal", "huggingface", "openai", "google"
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"

# ── Text chunking ────────────────────────────────────────────────────────────
CHUNK_SIZE = 1000                  # max chars per document chunk
CHUNK_OVERLAP = 100                # overlap between chunks

# ── Indexing pipeline ────────────────────────────────────────────────────────
BATCH_SIZE = 25_000

# ── Balancing & Synthetic ────────────────────────────────────────────────────
BALANCE_DECILES = False
ADD_SYNTHETIC_QUESTIONS = False
MIN_QUESTIONS_PER_DECILE = 200
MODEL_NAME = "gpt-4.1-nano"
SYNTHETIC_BATCH_SIZE = 500

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"✓ Config loaded: {QA_DATASETS} → {COLLECTION_NAME}")
print(f"  Strategy: {STRATEGY} | Embedding: {EMBEDDING_PROVIDER}")
print(f"  Chunk: {CHUNK_SIZE} chars (overlap {CHUNK_OVERLAP})")
print(f"  FAISS batch: {BATCH_SIZE:,}")
print(f"  Index dir: {FAISS_INDEX_DIR}")
print(f"  Balance: {BALANCE_DECILES} | Synthetic: {ADD_SYNTHETIC_QUESTIONS}")


/Users/cyro/Documents/VSC/PopularityBias/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✓ Config loaded: [] → wiki_full_f
  Strategy: vector | Embedding: modal
  Chunk: 1000 chars (overlap 100)
  FAISS batch: 25,000
  Index dir: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_f/faiss_index
  Balance: False | Synthetic: False


In [2]:
# ============================================================================
# STEP 1: Load QA Dataset (prepared by scripts/prepare_qa_dataset.py)
# ============================================================================
# To prepare the QA dataset from scratch, run:
#   python scripts/prepare_qa_dataset.py \
#       --qa-datasets natural_questions triviaqa \
#       --output data/wiki_1m_balanced_qa_b_nqtr/train_questions.parquet \
#       --balance
# ============================================================================

from scripts.prepare_qa_dataset import (
    prepare_qa_dataset,
)

if QUESTIONS_PATH.exists():
    print(f"Loading existing QA from {QUESTIONS_PATH}...")
    qa_df = pd.read_parquet(QUESTIONS_PATH)
    qa_df["wikipedia_id"] = pd.to_numeric(qa_df["wikipedia_id"], errors="coerce").astype(int)
    print(f"✓ Loaded {len(qa_df):,} questions")
else:
    print("No existing QA found — preparing from HuggingFace...")
    if not QA_DATASETS:
        logging.warning("No QA datasets specified! The index will be created without question-answer pairs, which may affect downstream evaluation.")
        qa_df = pd.DataFrame(columns=["question_text", "answer", "wikipedia_id"])
    else:
        qa_df = prepare_qa_dataset(
            qa_datasets=QA_DATASETS,
            popularity_dataset=POPULARITY_DATASET,
            output_path=QUESTIONS_PATH,
            balance=BALANCE_DECILES,
            cache_dir=CACHE_DIR,
        )

required_doc_ids = set(qa_df["wikipedia_id"])
print(f"\n✓ QA: {len(qa_df):,} questions | Documents needed: {len(required_doc_ids):,}")

if "decile" in qa_df.columns:
    print(f"Distribution:\n{qa_df['decile'].value_counts().sort_index()}")

display(qa_df.head())

Loading existing QA from /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_f/train_questions.parquet...
✓ Loaded 0 questions

✓ QA: 0 questions | Documents needed: 0


,question_text,answer,wikipedia_id


In [3]:
# ============================================================================
# STEP 2: Load Wikipedia (Only Required Documents)
# ============================================================================

if not WIKI_PARQUET_PATH.exists():
    print("\nLoading Wikipedia...")
    # Keep a reference to full dataset for retrieval
    full_wiki_ds = load_dataset(WIKIPEDIA_DATASET, WIKIPEDIA_VERSION, split="full", cache_dir=CACHE_DIR)
    full_wiki_ds = full_wiki_ds.select_columns(["wikipedia_id", "wikipedia_title", "text"])

    if N_RANDOM_SAMPLES is None:
        print("No random sampling configured, loading all documents...")
        wiki_ds = full_wiki_ds
    elif len(required_doc_ids) < N_RANDOM_SAMPLES:
        print(f"Sampling {N_RANDOM_SAMPLES:,} random documents from Wikipedia...")
        wiki_ds = full_wiki_ds.shuffle(seed=42).select(range(N_RANDOM_SAMPLES))
        print(f"✓ Loaded {len(wiki_ds):,} articles")
        
        # Check for missing required documents
        print("Checking for missing required documents...")
        existing_ids = set(int(i) for i in wiki_ds["wikipedia_id"] if i is not None)
        missing_ids = required_doc_ids - existing_ids
        print(f"✓ Missing: {len(missing_ids):,}")
        
        # Add missing docs if needed (OPTIMIZED - batched filtering)
        if missing_ids:
            print("  Fetching missing documents (this may take a moment)...")
            missing_ds = full_wiki_ds.filter(
                lambda batch: [int(i) in missing_ids for i in batch["wikipedia_id"]],
                batched=True,
                batch_size=10000,
                desc="Finding missing docs"
            )
            wiki_ds = concatenate_datasets([wiki_ds, missing_ds])
            print(f"✓ Added {len(missing_ds):,} missing docs. Total: {len(wiki_ds):,}")
    elif N_RANDOM_SAMPLES == 0:
        # Load only required documents (OPTIMIZED - batched filtering)
        print(f"Loading only {len(required_doc_ids):,} required documents...")
        if required_doc_ids:
            wiki_ds = full_wiki_ds.filter(
                lambda batch: [int(i) in required_doc_ids for i in batch["wikipedia_id"]],
                batched=True,
                batch_size=10000,
                desc="Loading required docs"
            )
            print(f"✓ Loaded {len(wiki_ds):,} articles")
        else:
            print("⚠️  No documents required, using empty dataset")
            wiki_ds = full_wiki_ds.select([])

    # Flatten KILT text structure - ALWAYS CHECK
    print("Normalizing text format...")

    def flatten_text(batch):
        return {
            "text": [
                "\n".join(t["paragraph"]) if isinstance(t, dict) and "paragraph" in t else str(t)
                for t in batch["text"]
            ]
        }

    # Apply to first item to check if needed
    needs_flattening = False
    if len(wiki_ds) > 0:
        sample_text = wiki_ds[0]["text"]
        if isinstance(sample_text, dict):
            needs_flattening = True

    if needs_flattening:
        wiki_ds = wiki_ds.map(flatten_text, batched=True, desc="Flattening text")
        print("✓ Text flattened")
    else:
        print("✓ Text already flat (or empty)")


In [4]:
# ============================================================================
# STEP 3: Add Popularity Metadata
# ============================================================================

if WIKI_PARQUET_PATH.exists():
    print("Wikipedia Parquet already exists, loading directly...")
else:
    print("Loading popularity data...")
    pop_ds = load_dataset(POPULARITY_DATASET, split="train+test", cache_dir=CACHE_DIR)

    # Detect columns
    cols = pop_ds.column_names
    id_col = "wikipedia_id" if "wikipedia_id" in cols else "id"
    rank_col = next((c for c in ["rank_avg", "avg_rank"] if c in cols), None)

    # Get needed IDs early
    needed_ids = set(int(x) for x in wiki_ds["wikipedia_id"] if x is not None)
    print(f"✓ Target documents: {len(needed_ids):,}")

    print("Processing popularity metadata (OPTIMIZED - Vectorized)...")

    # STEP 1: Load MINIMAL data first
    print("  Loading popularity scores...")
    pop_df_minimal = pop_ds.select_columns([id_col, "popularity_avg"]).to_pandas()
    pop_df_minimal[id_col] = pd.to_numeric(pop_df_minimal[id_col], errors='coerce').fillna(-1).astype(int)

    # STEP 2: FILTER EARLY - Keep only needed rows
    print(f"  Filtering from {len(pop_df_minimal):,} to {len(needed_ids):,} rows...")
    relevant_pop_df = pop_df_minimal[pop_df_minimal[id_col].isin(needed_ids)].copy()

    # STEP 3: Add rank column if needed
    if rank_col and rank_col in pop_ds.column_names:
        print("  Adding rank data for relevant documents only...")
        pop_ds_filtered = pop_ds.filter(
            lambda batch: [int(i) in needed_ids for i in batch[id_col]],
            batched=True,
            batch_size=10000,
            desc="Filtering popularity data",
        )
        rank_df = pop_ds_filtered.select_columns([id_col, rank_col]).to_pandas()
        rank_df[id_col] = rank_df[id_col].astype(int)
        relevant_pop_df = relevant_pop_df.merge(rank_df, on=id_col, how="left")

    # Standardize rank column name
    if rank_col:
        relevant_pop_df = relevant_pop_df.rename(columns={rank_col: "popularity_rank"})
    else:
        relevant_pop_df["popularity_rank"] = None

    # STEP 4: Build lookup dictionary (NO decile — boundaries are computed downstream)
    print("  Building optimized lookup...")
    pop_lookup = relevant_pop_df.set_index(id_col).to_dict(orient="index")

    # Cleanup
    del pop_df_minimal, pop_ds
    if rank_col:
        del pop_ds_filtered, rank_df
    gc.collect()

    print(f"✓ Built lookup with {len(pop_lookup):,} entries")

    # Merge with Wikipedia using batched map
    print("Merging metadata...")

    def merge_batch(batch):
        """Vectorized metadata merge — popularity only, no decile"""
        ids = [int(i) for i in batch["wikipedia_id"]]
        defaults = {"popularity_avg": None, "popularity_rank": None}
        meta_list = [pop_lookup.get(i, defaults) for i in ids]

        return {
            "popularity_avg": [m.get("popularity_avg") for m in meta_list],
            "popularity_rank": [m.get("popularity_rank") for m in meta_list],
        }

    wiki_ds_with_pop = wiki_ds.map(
        merge_batch,
        batched=True,
        batch_size=10_000,
        desc="Merging",
    )

    print(f"✓ Ready for indexing: {len(wiki_ds_with_pop):,} documents")

Wikipedia Parquet already exists, loading directly...


In [5]:
# ============================================================================
# STEP 3.5: (Optional) Synthetic Question Generation
# ============================================================================
# Handled by: python scripts/prepare_qa_dataset.py --generate-synthetic --corpus ...
# This cell is a no-op unless you want to re-generate inline.
# ============================================================================

if ADD_SYNTHETIC_QUESTIONS:
    from scripts.prepare_qa_dataset import generate_synthetic

    print("\n🤖 Generating synthetic questions...")
    qa_df = generate_synthetic(
        qa_df,
        corpus_path=WIKI_PARQUET_PATH,
        questions_per_decile=MIN_QUESTIONS_PER_DECILE,
        model_name=MODEL_NAME,
        batch_size=SYNTHETIC_BATCH_SIZE,
    )
    if BALANCE_DECILES:
        from scripts.prepare_qa_dataset import balance_by_decile
        qa_df = balance_by_decile(qa_df)
    print(f"✓ QA updated: {len(qa_df):,} questions")
else:
    print("✓ Skipping synthetic generation (ADD_SYNTHETIC_QUESTIONS = False)")

✓ Skipping synthetic generation (ADD_SYNTHETIC_QUESTIONS = False)


In [6]:
# ===========================================================================
# STEP 3.7: Data Sanitization & Save to Parquet (streaming to avoid RAM)
# ===========================================================================

print("\n🧹 Sanitizing data for FAISS indexing (streaming)...")
COLLECTION_ROOT.mkdir(parents=True, exist_ok=True)

print(f"\n💾 Saving documents to {WIKI_PARQUET_PATH}...")
if WIKI_PARQUET_PATH.exists():
    print(f"⚠️  Warning: {WIKI_PARQUET_PATH} already exists and existing will be used.")
else:
    writer = None
    n_docs = 0
    batch_size = 50_000

    # Calculate total batches for progress bar
    total_docs = len(wiki_ds_with_pop)
    total_batches = (total_docs + batch_size - 1) // batch_size

    pbar = tqdm(total=total_batches, desc="Saving to parquet", unit="batch")

    for batch in wiki_ds_with_pop.iter(batch_size=batch_size):
        batch_df = pd.DataFrame(batch)

        # Ensure numeric conversions happen after filling NaNs to avoid IntCastingNaNError
        batch_df["wikipedia_id"] = pd.to_numeric(batch_df["wikipedia_id"], errors="coerce").astype(int)

        # Use pandas nullable Float64 dtype for columns that may contain NaN
        batch_df["popularity_avg"] = pd.to_numeric(batch_df["popularity_avg"], errors="coerce").fillna(-1).astype("Float64")
        batch_df["popularity_rank"] = pd.to_numeric(batch_df["popularity_rank"], errors="coerce").astype("Float64")

        table = pa.Table.from_pandas(batch_df, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(WIKI_PARQUET_PATH, table.schema)
        writer.write_table(table)

        n_docs += len(batch_df)
        pbar.update(1)
        
        del batch_df, table
        gc.collect()

    pbar.close()

    if writer is not None:
        writer.close()

    print(f"  ✓ Saved {n_docs:,} documents ({WIKI_PARQUET_PATH.stat().st_size / 1e9:.2f} GB)")
    del wiki_ds_with_pop, wiki_ds, full_wiki_ds
    gc.collect()


# ── Save training questions ──────────────────────────────────────────────────
print(f"Saving training questions to {QUESTIONS_PATH}...")
if QUESTIONS_PATH.exists():
    print(f"⚠️  Warning: {QUESTIONS_PATH} already exists and existing will be used.")
else:
    qa_df.to_parquet(QUESTIONS_PATH, index=False, engine="pyarrow")
    print(f"  ✓ Saved {len(qa_df):,} questions")

# ── Free memory — everything is on disk now ──────────────────────────────────
del qa_df
gc.collect()


🧹 Sanitizing data for FAISS indexing (streaming)...

💾 Saving documents to /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_f/wiki_corpus.parquet...
⚠️  Warning: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_f/wiki_corpus.parquet already exists and existing will be used.
Saving training questions to /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_f/train_questions.parquet...
⚠️  Warning: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_f/train_questions.parquet already exists and existing will be used.


40

In [ ]:

# ============================================================================
# STEP 4: Create FAISS Index (streaming from Parquet)
# ============================================================================

config = IndexingConfig(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    batch_size=BATCH_SIZE,
    embedding_provider=EMBEDDING_PROVIDER,
    embedding_model=EMBEDDING_MODEL,
    trust_remote_code=True,
    use_progress=True,
)

service = FaissRagService(
    config=config,
    strategy=STRATEGY,
    distance_strategy="cosine",
)

SKIP_ROWS = 0

print(f"👉 Starting indexing from row {SKIP_ROWS:,}")

index, num_chunks = service.index_from_parquet_batches(
    parquet_path=WIKI_PARQUET_PATH,
    text_field="text",
    metadata_fields=["wikipedia_id", "wikipedia_title", "popularity_avg", "popularity_rank"],
    collection_name=str(FAISS_INDEX_DIR),
    progress_bar=True,
    batch_size=BATCH_SIZE,
    skip_rows=SKIP_ROWS,
    checkpoint=True,
)

print(f"\n✅ DONE")
print(f"  Chunks indexed: {num_chunks:,}")
print(f"  Index saved to: {FAISS_INDEX_DIR}")
print(f"  Strategy: {STRATEGY}")
print(f"\nNext: Run rag_evaluation.ipynb")


INFO - FAISS vector strategy ready
INFO - Indexing 5,903,530 rows (of 5,903,530 total) | parquet_batch=25,000


👉 Starting indexing from row 0


Indexing:   0%|          | 0/5903530 [00:00<?, ?row/s]INFO - [Prepare] 25,000 rows from parquet
INFO - [Prepare] 137,472 chunks (from 25,000 rows)
INFO - [Embed] 137,472 chunks...
INFO - [Prepare] 25,000 rows from parquet
INFO - [Prepare] 104,721 chunks (from 25,000 rows)
INFO - [Prepare] 25,000 rows from parquet
INFO - [Prepare] 98,789 chunks (from 25,000 rows)
INFO - [Prepare] 25,000 rows from parquet
INFO - [Prepare] 103,300 chunks (from 25,000 rows)
INFO - [Insert] 137,472 docs → FAISS
INFO - [Embed] 104,721 chunks...
INFO - [Prepare] 25,000 rows from parquet
INFO - [Prepare] 110,074 chunks (from 25,000 rows)
Indexing:   0%|          | 25000/5903530 [05:26<21:21:26, 76.46row/s]INFO - [Insert] ✓ +137,472 chunks / 25,000 rows (cumulative: 137,472 chunks, 25,000 rows)
INFO - [Checkpoint] Saving index at 25,000 rows...
INFO - FAISS index saved to /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_f/faiss_index/faiss (137,472 vectors)
INFO - [Checkpoint] Saved at 25,000 rows (137,4